In [11]:
from pathlib import Path
import os

repo_root = Path("/home/thardy/elefanto/ConsciousnessTeam_Data/SOUNDLEAD")
os.chdir(repo_root)

In [19]:
"""
Study 3 (SOUNDIAGNOSE patients, PSL + HCL): EGI .mff -> BrainVision (.vhdr/.vmrk/.eeg).

Format conversion only: no filtering, no channel rejection, no re-referencing, no resampling.
Event curation follows the MATLAB pipeline (14 valid codes, P16/P18/P24 fixes).

Output per patient (OUT_DIR/Pxx/):
    Pxx.vhdr / .vmrk / .eeg   all EEG channels, native sfreq, Cz/VREF reference
    Pxx-dig.fif               electrode positions  -> mne.channels.read_dig_fif
    Pxx_trials.tsv            one row per valid trial
    Pxx_info.json             metadata + QC
OUT_DIR/participants.tsv

vmrk markers:
    Stimulus/S111 ... S164    valid trials (code = Dxyz without the 'D': x=1, y=SNR, z=stim 1:A 2:E 3:II 4:OO)
    Comment/dropped_Dxyz      valid code but excluded by the per-patient fixes
    Comment/<code>            all other EGI events, kept untouched (DI60, DIN4, sync, boundaries...)

Original event codes:
D111 = Asnr1 — 80 repetitions across 4 blocks
D112 = Esnr1
D121/D122 = A/E snr2
D131/D132 = A/E snr3
D141/D142 = A/E snr4
D151/D152 = A/E snr5
D161/D162 = A/E snr6
D163/D164 = II/OO

pip install mne pybv defusedxml scipy pandas   (+ mat73 if behaviour files are v7.3)
"""

import json
import warnings
from pathlib import Path

import mne
import numpy as np
import pandas as pd
from scipy.io import loadmat

RAW_DIR = repo_root / "Study3" / "Study3_mff"
OUT_DIR = repo_root / "Study3"

TARGET_CODES = ["D111", "D112", "D121", "D122", "D131", "D132", "D141",
                "D142", "D151", "D152", "D161", "D162", "D163", "D164"]
N_TRIALS = 1040
TRIALS_PER_BLOCK = 260

HCL = {6, 7, 8, 26, 27, 28, 29, 30, 31, 32}
GROUPS = {
    "CS": [12, 21, 7, 27, 30],
    "MCS": [5, 11, 14, 15, 22, 23, 24, 25, 6, 26, 28, 29, 32],
    "VS": [9, 10, 13, 16, 17, 18, 19, 20, 8, 31],
}
EXCLUDED_BY_MARTINA = {12, 21, 22, 25, 10, 13, 17}
NO_FRT = {7, 16, 18, 20, 24, 25, 27, 28, 29, 31, 32} # Patients with stim-locked high face movements during the experiment, so frontal electrodes are noisy.
BEHAV_AVAILABLE = {5, 7, 8, 9, 15, 16, 24, 26, 27, 28, 29}

# python slices over the list of valid-code events (unvalid ones are removed by hand by Julie for these 3 participants)
TRIAL_SELECTION = {
    16: [slice(2, None)],
    18: [slice(38, None)],
    24: [slice(0, 780), slice(845, 1105)],
}

ALL_PATIENTS = sorted(p for ps in GROUPS.values() for p in ps)
GROUP_OF = {p: g for g, ps in GROUPS.items() for p in ps}


def load_mff(pat_dir):
    files = sorted(f for f in pat_dir.glob("*.mff") if not f.name.startswith("."))
    if not files:
        raise FileNotFoundError(f"no .mff in {pat_dir}")
    raws = []
    for f in files:
        r = mne.io.read_raw_egi(f, preload=False, events_as_annotations=True, verbose="error")
        r.pick("eeg")
        raws.append(r)
    sfreqs = {r.info["sfreq"] for r in raws}
    if len(sfreqs) > 1:
        raise ValueError(f"parts have different sfreq: {sfreqs}")
    raw = mne.concatenate_raws(raws, verbose="error") if len(raws) > 1 else raws[0]
    return raw, files


def onsets_in_samples(raw, onsets):
    ann = raw.annotations
    rel = np.asarray(onsets) - (raw.first_time if ann.orig_time is not None else 0.0)
    return raw.time_as_index(rel).astype(int)


def curate_events(raw, ip):
    ann = raw.annotations
    desc = np.array([str(d).replace(",", ";") for d in ann.description], dtype=object)

    target_idx = np.flatnonzero(np.isin(desc, TARGET_CODES))
    kept = np.concatenate([target_idx[s] for s in TRIAL_SELECTION.get(ip, [slice(None)])])
    dropped = np.setdiff1d(target_idx, kept)

    new_desc = desc.copy()
    for i in kept:
        new_desc[i] = f"Stimulus/S{desc[i][1:]}"
    for i in dropped:
        new_desc[i] = f"Comment/dropped_{desc[i]}"

    raw.set_annotations(mne.Annotations(
        onset=ann.onset, duration=ann.duration,
        description=[str(d) for d in new_desc], orig_time=ann.orig_time,
    ))

    codes = desc[kept]
    pos = np.arange(1, len(kept) + 1)
    trials = pd.DataFrame({
        "trial": pos,
        "onset_sample": onsets_in_samples(raw, ann.onset[kept]),
        "onset_s": ann.onset[kept] - (raw.first_time if ann.orig_time is not None else 0.0),
        "code": codes,
        "block": np.ceil(pos / TRIALS_PER_BLOCK).astype(int),
        "snr": [int(c[2]) for c in codes],
        "stim": [int(c[3]) for c in codes],
    })
    trials["vowel"] = trials["stim"].map({1: "A", 2: "E", 3: "II", 4: "OO"})
    trials["task_relevant"] = trials["stim"].isin([3, 4])

    other, counts = np.unique(desc[~np.isin(desc, TARGET_CODES)], return_counts=True)
    qc = {
        "n_valid_code_events": int(len(target_idx)),
        "n_trials_kept": int(len(kept)),
        "n_dropped_valid_code_events": int(len(dropped)),
        "missing_codes": sorted(set(TARGET_CODES) - set(codes)),
        "other_event_codes": {str(o): int(c) for o, c in zip(other, counts)},
    }
    if len(kept) != N_TRIALS:
        warnings.warn(f"P{ip:02d}: {len(kept)} trials kept (expected {N_TRIALS})")
    if qc["missing_codes"]:
        warnings.warn(f"P{ip:02d}: missing codes {qc['missing_codes']}")
    return trials, qc


def _mat_to_dict(t):
    if isinstance(t, dict):
        return {k: np.asarray(v).ravel() for k, v in t.items()}
    if isinstance(t, (list, np.ndarray)) and len(t) and isinstance(t[0], dict):
        return {k: np.array([row[k] for row in t]).ravel() for k in t[0]}
    raise TypeError("`trials` is neither a struct nor a struct array (MATLAB table? re-save as struct)")


def load_behaviour(pat_dir):
    for f in sorted(pat_dir.glob("S*.mat")):
        try:
            m = loadmat(f, simplify_cells=True)
        except NotImplementedError:
            import mat73
            m = mat73.loadmat(f)
        if "trials" in m:
            return _mat_to_dict(m["trials"]), f.name
    warnings.warn(f"{pat_dir.name}: no S*.mat containing `trials`, behaviour check skipped")
    return None, None


def check_behaviour(trials, beh, ip):
    out = {}
    for eeg_col, beh_col, conv in [("code", "stimulus_trigger", lambda c: int(c[1:])),
                                   ("block", "block", int), ("snr", "snr_num", int),
                                   ("stim", "stimulus_num", int)]:
        if beh_col not in beh:
            out[beh_col] = "absent"
            continue
        e = np.array([conv(v) for v in trials[eeg_col]])
        b = np.asarray(beh[beh_col]).astype(int)
        if len(e) != len(b):
            out[beh_col] = f"length mismatch (eeg {len(e)}, behav {len(b)})"
        elif not np.array_equal(e, b):
            out[beh_col] = f"mismatch, first at trial {int(np.flatnonzero(e != b)[0]) + 1}"
        else:
            out[beh_col] = "ok"
        if out[beh_col] != "ok":
            warnings.warn(f"P{ip:02d}: behaviour {beh_col}: {out[beh_col]}")
    return out


def convert_patient(ip, raw_dir=RAW_DIR, out_dir=OUT_DIR):
    sub = f"P{ip:02d}"
    pat_dir = raw_dir / f"P{ip}"
    out = out_dir / sub
    out.mkdir(parents=True, exist_ok=True)
    print(f"### {sub} ###")

    raw, files = load_mff(pat_dir)
    boundaries = sorted({float(o) for o, d in zip(raw.annotations.onset, raw.annotations.description)
                         if "boundary" in d.lower() or "acq_skip" in d.lower()})
    trials, qc = curate_events(raw, ip)

    beh_check, beh_file = None, None
    if ip in BEHAV_AVAILABLE:
        beh, beh_file = load_behaviour(pat_dir)
        if beh is not None:
            beh_check = check_behaviour(trials, beh, ip)

    mne.export.export_raw(out / f"{sub}.vhdr", raw, fmt="brainvision", overwrite=True, verbose="error")
    montage = raw.get_montage()
    if montage is not None:
        montage.save(out / f"{sub}-dig.fif", overwrite=True, verbose="error")
    trials.to_csv(out / f"{sub}_trials.tsv", sep="\t", index=False)

    info = {
        "participant": sub,
        "study": 3,
        "site": "HCL" if ip in HCL else "PSL",
        "group": GROUP_OF[ip],
        "excluded_by_martina": ip in EXCLUDED_BY_MARTINA,
        "no_frt": ip in NO_FRT,
        "sfreq": raw.info["sfreq"],
        "n_channels": len(raw.ch_names),
        "ch_names": raw.ch_names,
        "duration_s": raw.times[-1],
        "reference": "Cz (EGI VREF), as recorded",
        "mff_files": [f.name for f in files],
        "segment_boundaries_s": boundaries,
        "trial_selection": [str(s) for s in TRIAL_SELECTION.get(ip, [])],
        "behaviour_file": beh_file,
        "behaviour_check": beh_check,
        **qc,
    }
    (out / f"{sub}_info.json").write_text(json.dumps(info, indent=2))
    return info


def run(patients=None, raw_dir=RAW_DIR, out_dir=OUT_DIR):
    patients = ALL_PATIENTS if patients is None else patients
    raw_dir, out_dir = Path(raw_dir), Path(out_dir)
    rows, failed = [], {}
    for ip in patients:
        try:
            info = convert_patient(ip, raw_dir, out_dir)
            rows.append({k: info[k] for k in ("participant", "site", "group", "excluded_by_martina",
                                              "no_frt", "sfreq", "n_channels", "n_trials_kept")})
        except Exception as e:
            failed[ip] = repr(e)
            print(f"!!! P{ip:02d} failed: {e!r}")

    if rows:
        ptsv = out_dir / "participants.tsv"
        new = pd.DataFrame(rows)
        if ptsv.exists():
            old = pd.read_csv(ptsv, sep="\t")
            new = pd.concat([old[~old.participant.isin(new.participant)], new]).sort_values("participant")
        new.to_csv(ptsv, sep="\t", index=False)
    if failed:
        print("Failed:", failed)
    return rows, failed

In [18]:
patients = [12]          # ALL_PATIENTS or e.g. [15, 16, 24] to test a few first

rows, failed = run(patients, RAW_DIR, OUT_DIR)

pd.read_csv(OUT_DIR / "participants.tsv", sep="\t")

### P12 ###
categories.xml not found or of wrong type. `Epoch.name` will default to "epoch" for all epochs.
!!! P12 failed: PermissionError(13, 'Permission denied')
Failed: {12: "PermissionError(13, 'Permission denied')"}


,participant,site,group,excluded_by_martina,no_frt,sfreq,n_channels,n_trials_kept
0,P05,PSL,MCS,False,False,500.0,257,1040
1,P06,HCL,MCS,False,False,500.0,129,1040
2,P07,HCL,CS,False,True,500.0,129,1040
3,P08,HCL,VS,False,False,500.0,129,1040
4,P09,PSL,VS,False,False,500.0,257,1040
5,P10,PSL,VS,True,False,500.0,257,1040
6,P11,PSL,MCS,False,False,500.0,257,1040
7,P12,PSL,CS,True,False,500.0,257,1040
8,P13,PSL,VS,True,False,500.0,257,1040
9,P14,PSL,MCS,False,False,500.0,257,1040


In [20]:
import traceback, subprocess

try:
    convert_patient(12, RAW_DIR, OUT_DIR)
except Exception:
    traceback.print_exc()

for p in [RAW_DIR / "P12", *sorted((RAW_DIR / "P12").glob("*.mff")), OUT_DIR / "P12"]:
    print(subprocess.run(["ls", "-la", str(p)], capture_output=True, text=True).stdout or f"{p}: does not exist")

### P12 ###
categories.xml not found or of wrong type. `Epoch.name` will default to "epoch" for all epochs.
total 20
drwxrwxrwx  3 www-data users 4096 29 oct.  18:24 .
drwxrwxrwx 30 www-data users 4096 29 oct.  18:24 ..
-rw-rw-rw-  1 www-data users 6148 18 janv.  2025 .DS_Store
drwxrwxrwx  3 www-data users 4096 29 oct.  16:00 SoDi_P12_SD.mff

total 1692632
drwxrwxrwx 3 www-data users       4096 29 oct.  16:00 .
drwxrwxrwx 3 www-data users       4096 29 oct.  18:24 ..
drwxrwxrwx 2 www-data users       4096 29 oct.  16:00 Contents
-rw-rw-rw- 1 www-data users      62635 21 juin   2019 coordinates.xml
-rw-rw-rw- 1 www-data users        511 21 juin   2019 epochs.xml
-rw-rw-rw- 1 www-data users        334 21 juin   2019 epochs.xml.bak
-rw-rw-rw- 1 www-data users        702 21 juin   2019 Events_Calibration.xml
-rw-rw-rw- 1 www-data users    1838478 21 juin   2019 Events_DIN_1.xml
-rw-rw-rw- 1 www-data users      33483 21 juin   2019 info1.xml
-rw-rw-rw- 1 www-data users      22390 21 juin   

Traceback (most recent call last):
  File "/tmp/ipykernel_510653/1889305285.py", line 4, in <module>
    convert_patient(12, RAW_DIR, OUT_DIR)
  File "/tmp/ipykernel_510653/1413411494.py", line 200, in convert_patient
    mne.export.export_raw(out / f"{sub}.vhdr", raw, fmt="brainvision", overwrite=True, verbose="error")
  File "<decorator-gen-294>", line 10, in export_raw
  File "/home/thardy/soundmodelenv/lib/python3.11/site-packages/mne/export/_export.py", line 87, in export_raw
    _export_raw(fname, raw, overwrite)
  File "/home/thardy/soundmodelenv/lib/python3.11/site-packages/mne/export/_brainvision.py", line 165, in _export_raw
    _export_mne_raw(raw=raw, fname=fname, overwrite=overwrite)
  File "/home/thardy/soundmodelenv/lib/python3.11/site-packages/mne/export/_brainvision.py", line 92, in _export_mne_raw
    write_brainvision(
  File "/home/thardy/soundmodelenv/lib/python3.11/site-packages/pybv/io.py", line 320, in write_brainvision
    _write_bveeg_file(
  File "/home/thard

In [21]:
import subprocess

print(subprocess.run(["df", "-h", str(OUT_DIR)], capture_output=True, text=True).stdout)
(OUT_DIR / "P12" / "P12.eeg").unlink(missing_ok=True)
test = OUT_DIR / "P12" / "_size_test.bin"
chunk = b"\0" * 100_000_000
try:
    with open(test, "wb") as f:
        for i in range(25):
            f.write(chunk)
            f.flush()
    print("2.5 GB written: no size limit")
except OSError as e:
    print(f"failed after {test.stat().st_size / 1e9:.2f} GB: {e!r}")
finally:
    test.unlink(missing_ok=True)

Sys. de fichiers          Taille Utilisé Dispo Uti% Monté sur
172.20.109.124:/home/octo   203T    6,6T  187T   4% /srv/elefanto

2.5 GB written: no size limit
